In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch09-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "09-03-regularization.py"

# 限制指紋係數後，哪些模型的驗證誤差較小？
第一次執行需連網取得資料。分析程式在獨立的 Python 程序執行。練習程式可接在寫入暫存檔的儲存格末尾，再執行該格及最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem

SEED = 42
np.random.seed(SEED)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
options = dict(featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
               transformers=[], data_dir=str(cache), reload=False)
# ESOL：1,128 筆實測水溶解度 log10(mol/L)，DOI 10.1021/ci034243x。
tasks, (train, valid, test), _ = dc.molnet.load_delaney(**options)
assert tasks == ["measured log solubility in mols per litre"]
assert all(np.all(split.w > 0) for split in (train, valid, test))
y_train, y_valid, y_test = (split.y.ravel() for split in (train, valid, test))
print("split sizes:", len(train), len(valid), len(test))
from rdkit.Chem import rdFingerprintGenerator
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error

def fingerprints(split):
    return np.asarray([generator.GetFingerprintAsNumPy(Chem.MolFromSmiles(smiles))
                       for smiles in split.ids], dtype=float)

X_train, X_valid, X_test = (fingerprints(split) for split in (train, valid, test))
candidates = {
    "Ridge": [Ridge(alpha=a) for a in [0.1, 1, 10, 100]],
    "Lasso": [Lasso(alpha=a, max_iter=20000) for a in [0.001, 0.01, 0.1, 1]],
    "Elastic Net": [ElasticNet(alpha=a, l1_ratio=r, max_iter=20000)
                    for a in [0.001, 0.01, 0.1] for r in [0.25, 0.5, 0.75]],
}
rows = []
for family, models in candidates.items():
    for model in models:
        model.fit(X_train, y_train)
        mae = mean_absolute_error(y_valid, model.predict(X_valid))
        rows.append((family, model, mae))
        print(f"{family:12s} {model.alpha:6g} {getattr(model, 'l1_ratio', '-')} "
              f"{mae:.4f} {np.count_nonzero(model.coef_)}")
best_family, best, best_mae = min(rows, key=lambda row: row[2])
print("selected, test MAE:", best_family, best, mean_absolute_error(y_test, best.predict(X_test)))
fig, axes = plt.subplots(1, 3, figsize=(9.1, 3.5), sharey=True, layout="constrained")
for ax, (family, models) in zip(axes, candidates.items()):
    records = [row for row in rows if row[0] == family]
    ratios = [0.25, 0.5, 0.75] if family == "Elastic Net" else [None]
    for ratio in ratios:
        subset = [(model.alpha, mae) for _, model, mae in records
                  if ratio is None or model.l1_ratio == ratio]
        alpha, errors = zip(*subset)
        ax.plot(alpha, errors, marker="o", label=f"r = {ratio}")
    ax.set(xscale="log", xlabel="alpha", title=family)
    if family == "Elastic Net":
        ax.legend(fontsize=8)
axes[0].set_ylabel("Validation MAE (log S)")
fig.savefig(figure_dir / "09-03-regularization.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-03-regularization.png")))